In [ ]:

# Global imports & style (hidden)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
# Palette (WCAG-minded; avoid red/green pairing)
BLUE, ORNG, PURP, GREY, TXT, GRID, BG = '#1f77b4', '#ff7f0e', '#7b3294', '#B0B0B0', '#222', '#e8e8e8', 'white'
plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': BG, 'axes.edgecolor':'#ccc',
    'axes.grid': True, 'grid.color': GRID, 'grid.alpha': 0.3,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.labelsize': 11,
    'font.size': 11, 'legend.frameon': False
})

def ensure_year_on_dates(ax):
    ax.xaxis.set_major_formatter(DateFormatter('%b %d %Y'))
print('✅ Style ready')


# HERO — Artist‑First Pledge
We invest in music that moves people. This dashboard prioritizes artists and audience experience: clarity over clutter, action over vanity, and accessibility for all.

## S0 — Budget Playbook
- Pick a lane first: Growth (new), Sustain (active), or Revive (dormant).
- Budget follows the choice; charts help validate the bet.
- Color is used sparingly to spotlight the decision; grey carries context.

## S1 — Strategy Pros/Cons (A/B/C)
- A. Growth: fast tests, thumbnail/title sprints, release cadence.
- B. Sustain: compound wins, keep fans warm, feature timing.
- C. Revive: catalog moments, collab hooks, re‑edits.

## H2 — Artist Spotlight
Quotes from socials + days ≥55 this year (if data present).

In [ ]:

# Spotlight card — per-artist summary and days ≥55 if available
if 'vids_safe' in globals():
    df = vids_safe.copy()
    df['published_at'] = pd.to_datetime(df['published_at'], errors='coerce')
    df['pub_date'] = df['published_at'].dt.floor('D')

    # Base stats
    base = df.groupby('artist_name').agg(
        videos=('video_id','nunique'),
        views=('view_count','sum')
    ).sort_values('views', ascending=False)

    # Days ≥55 via momentum_daily if present
    days55 = None
    try:
        if 'momentum_daily' in globals():
            md = momentum_daily.copy()
            md['date'] = pd.to_datetime(md['date']).dt.floor('D')
            hits = md.loc[md['momentum_score']>=55, ['video_id','date']]
            amap = df[['video_id','artist_name']].drop_duplicates()
            days55 = hits.merge(amap, on='video_id', how='left')                         .groupby('artist_name')['date'].nunique().rename('days_55')
    except Exception:
        days55 = None

    spotlight = base
    if days55 is not None:
        spotlight = spotlight.join(days55, how='left').fillna({'days_55':0}).astype({'days_55':'int64'})

    display(spotlight.head(6))
else:
    print('ℹ️ vids_safe not found — run metrics cell')


## Chart 1.1 — Roster Overview (per‑artist key stats)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CHART 1.1: Artist Roster Overview
# ═══════════════════════════════════════════════════════════════════════════
TEXT="#FFFFFF"

# Aggregate metrics by artist
artist_summary = vids.groupby('artist_name').agg({
    'video_id': 'count',
    'view_count': 'sum',
    'like_count': 'sum',
    'comment_count': 'sum',
    'views_per_day': 'mean',
    'like_rate': 'mean',
}).rename(columns={
    'video_id': 'videos',
    'view_count': 'total_views',
    'like_count': 'total_likes',
    'comment_count': 'total_comments',
    'views_per_day': 'avg_views_per_day',
    'like_rate': 'avg_like_rate',
}).round(2)

artist_summary = artist_summary.sort_values('total_views', ascending=False)

# Display as table
from IPython.display import display, HTML

html_table = artist_summary.to_html(classes='table table-striped', border=0)
display(HTML(f"""          
<div style="padding:20px;border:2px solid {PALETTE[0]};border-radius:12px;background:#f9f9f9,color:{TEXT};">
    <h3>🎵 Artist Roster Overview ({ARTIST_COUNT} Artists)</h3>
    {html_table}
</div>
"""))

print(f"\n✅ Roster summary: {ARTIST_COUNT} artists, {artist_summary['videos'].sum():.0f} total videos")

## Chart 1.2 — Engagement Patterns by Artist (interactive)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# DATA PREPARATION + SAFE COPIES
# ═══════════════════════════════════════════════════════════════════════════

# Create working copies
vids = videos_df.copy()
comments = comments_df.copy()

# Derived video features
vids["age_days"] = (END_DATE - vids["published_at"]).dt.days.clip(lower=1)
vids["views_per_day"] = (vids["view_count"] / vids["age_days"]).replace([np.inf, np.nan], 0.0)
vids["like_rate"] = (vids["like_count"] / vids["view_count"].replace(0, np.nan)).fillna(0.0).clip(0,1)
vids["comment_rate"] = (vids["comment_count"] / vids["view_count"].replace(0, np.nan)).fillna(0.0).clip(0,1)
vids["publish_week"] = vids["published_at"].dt.to_period("W").dt.to_timestamp()
vids["publish_month"] = vids["published_at"].dt.to_period("M").dt.to_timestamp()
vids["publish_hour"] = vids["published_at"].dt.hour
vids["publish_dow"] = vids["published_at"].dt.day_name()

# Comment features
if 'text' in comments.columns:
    comments["comment_length"] = comments["text"].str.len().fillna(0)

# ── Safe copies for downstream visuals ──────────────────────────────────────
vids_safe = vids.copy()
vids_safe["published_at"] = pd.to_datetime(vids_safe["published_at"], errors="coerce")
vids_safe = vids_safe.dropna(subset=["published_at"])
vids_safe["pub_date"] = vids_safe["published_at"].dt.floor("D")

comments_safe = None
if "comments_df" in globals():
    comments_safe = comments_df.copy()
elif "comments" in globals():
    comments_safe = comments.copy()

if comments_safe is not None:
    comments_safe["published_at"] = pd.to_datetime(comments_safe["published_at"], errors="coerce")
    comments_safe = comments_safe.dropna(subset=["published_at"])
    comments_safe["pub_date"] = comments_safe["published_at"].dt.floor("D")

print(f"✅ Data prepared: {len(vids):,} videos with derived features")
print(f"   Age range: {vids['age_days'].min():.0f} - {vids['age_days'].max():.0f} days")
print(f"   Views/day range: {vids['views_per_day'].min():.1f} - {vids['views_per_day'].max():.1f}")
print("✅ Metrics prepared (comments optional).")

## Chart 1.3 — Performance Trends (Exec view)

In [ ]:
# ---- Choose how Chart 1.3 renders ----
# "exec"    = Beauty Pass (weekly, winsorized, clean)
# "both"    = Exec slide + Analyst appendix
# "analyst" = Analyst-only (daily, full rigor)
VIEW_MODE = "exec"
print(f"Chart 1.3 VIEW_MODE = {VIEW_MODE}")

## Chart 1.3a — Analyst: Base‑100 + MAD outliers

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 4 — Visualization Helpers (direct labels + date-year guard)
# ═══════════════════════════════════════════════════════════════════════════
from textwrap import shorten

def clean_title(ax, text: str):
    ax.set_title(shorten(text, width=140, placeholder="…"), fontweight="bold", color=TXT)

def add_value_labels(ax, fmt="{:,.0f}", padding=4, orientation="vertical"):
    """Label bar values directly (no legend scan). orientation: 'vertical'|'horizontal'"""
    for c in ax.containers:
        for bar in c:
            if orientation == "vertical":
                val = bar.get_height()
                if np.isnan(val):
                    continue
                ax.text(bar.get_x() + bar.get_width()/2, val, fmt.format(val),
                        ha="center", va="bottom", fontsize=10, color=TXT)
            else:
                val = bar.get_width()
                if np.isnan(val):
                    continue
                ax.text(val, bar.get_y() + bar.get_height()/2, fmt.format(val),
                        ha="left", va="center", fontsize=10, color=TXT)

def direct_line_label(ax, x, y, text, dx=6, dy=0):
    ax.annotate(text, xy=(x[-1], y[-1]), xytext=(dx,dy), textcoords="offset points",
                ha="left", va="center", fontsize=10, color=TXT, fontweight="bold")

def ensure_year_on_dates(ax):
    """Force year on dates (e.g., 'Jan 01 2025')."""
    ax.xaxis.set_major_formatter(DateFormatter("%b %d %Y"))

def robust_modified_z(series: pd.Series):
    """Modified Z (median/MAD); NaN-safe."""
    x = series.dropna()
    if x.empty:
        return pd.Series(index=series.index, dtype=float)
    med = x.median()
    mad = (x - med).abs().median()
    if mad == 0:
        return pd.Series(0.0, index=series.index)
    return 0.6745 * (series - med) / mad

print("✅ Helpers ready (direct labels, year guard, robust z)")

## Chart 2.1 — Threshold Comparison (55 vs 75) + Video‑Day explainer

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CONFIG & HUMAN-REVIEW GATES
# ═══════════════════════════════════════════════════════════════════════════

# --- Artist Roster (HUMAN-REVIEW) ---
ARTISTS_OVERRIDE = [
    # Optional roster override — edit or leave empty to infer from data.
    # "hicorook", "Flyana Boss", "Raiche", "BiC Fizzle", "COBRAH", "re6ce"
]

# --- Approval Flags (HUMAN-REVIEW) ---
HUMAN_REVIEW_APPROVED = {
    "artists_override": False,   # set True after you review ARTISTS_OVERRIDE
    "budget_inputs":    False,   # set True after you set BUDGET amounts
}

# --- Thresholds ---
THRESHOLDS = {
    "legacy":       75,  # for comparison plots (historical)
    "pre_breakout": 55,  # BLUE highlight + invest-gradually signal
    "breakout":     60,  # legacy episode computation (kept for reference)
}


# --- Episode Detection Controls ---
EPISODE_THRESHOLD = THRESHOLDS.get("legacy", THRESHOLDS["breakout"])  # default to the stricter 75 viral bar
PREWARN_LOWER = THRESHOLDS["pre_breakout"]
PREWARN_MAX_DAYS = 30  # cap early-warning lookback so alerts stay tactical
SCMTV_BREAKOUT_FLOOR = max(THRESHOLDS["pre_breakout"], 60)  # require community energy, not just static KPIs

# --- Budget Parameters (HUMAN-REVIEW REQUIRED) ---
BUDGET = {
    # >>>>> HUMAN REVIEW REQUIRED — set your real amounts/percentages <<<<<
    "tier_55_pct":      None,  # e.g., 0.20 for 20% of pool
    "tier_75_pct":      None,  # e.g., 0.35
    "cap_per_artist":   None,  # e.g., 1500.0 (USD)
}

# --- Analysis Period ---
import pandas as pd
from datetime import datetime, timedelta

END_DATE = pd.Timestamp.now().normalize()
START_DATE = END_DATE - timedelta(days=90)  # 90-day analysis window

# --- Human-Review Badge (with forced black text) ---
def _hr_badge():
    msgs = []
    if ARTISTS_OVERRIDE and not HUMAN_REVIEW_APPROVED["artists_override"]:
        msgs.append("Artist roster override present but not approved.")
    if any(BUDGET[k] in (None, 0) for k in ["tier_55_pct","tier_75_pct","cap_per_artist"]) \
       and not HUMAN_REVIEW_APPROVED["budget_inputs"]:
        msgs.append("Budget inputs not approved or unset.")
    if msgs:
        from IPython.display import HTML, display
        html = "<br>".join(f"<span style='color:#000000 !important;'>• {m}</span>" for m in msgs)
        display(HTML(f'''
        <style>
        .warning-box, .warning-box * {{
            color: #000000 !important;
        }}
        </style>
        <div class="warning-box" style="padding:14px;border:3px solid #d95f02;border-radius:12px;background:#fff3e6;color:#000000 !important;">
          <b style="color:#000000 !important;">⚠️  HUMAN-REVIEW REQUIRED</b><br>{html}
        </div>
        '''))
    else:
        from IPython.display import HTML, display
        display(HTML(f'''
        <style>
        .success-box, .success-box * {{
            color: #000000 !important;
        }}
        </style>
        <div class="success-box" style="padding:14px;border:2px solid #1b9e77;border-radius:12px;background:#e6fff9;color:#000000 !important;">
          <b style="color:#000000 !important;">✅ All human-review gates approved</b>
        </div>
        '''))

_hr_badge()

print(f"\n📅 Analysis Period: {START_DATE.date()} → {END_DATE.date()}")
print(f"🎯 Thresholds: Pre-breakout={PREWARN_LOWER}, Breakout episodes={EPISODE_THRESHOLD}, Historical baseline={THRESHOLDS['breakout']}")


## Chart 2.2 — Threshold Impact vs Ops Capacity (cumulative video‑days)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 15 — Chart 2.1: Threshold comparison (55 vs 75) + Video-Day explainer
# ═══════════════════════════════════════════════════════════════════════════
from IPython.display import display, Markdown

THRESHOLDS = THRESHOLDS if "THRESHOLDS" in globals() else {"pre_breakout":55, "legacy":75, "breakout":60}
if "momentum_daily" not in globals():
    raise RuntimeError("momentum_daily missing — run momentum build first.")

display(Markdown("""
### 📊 **Video-Day (ELI-8)**
**Video-day** = a video meets the threshold for one calendar day.  
If 3 qualify Monday and 2 Tuesday → **5 video-days**.  
More video-days = more moments to activate.
"""))

def _daily_hits(d, t):
    return d.loc[d["momentum_score"]>=t].groupby("date")["video_id"].nunique().rename(f"videos_at_{int(t)}")

d55 = _daily_hits(momentum_daily, THRESHOLDS["pre_breakout"])
d75 = _daily_hits(momentum_daily, THRESHOLDS["legacy"])
q = pd.concat([d55, d75], axis=1).fillna(0).reset_index().sort_values("date")

# ✅ named columns (no positional iloc)
q["incremental_videos"] = q[f"videos_at_{int(THRESHOLDS['pre_breakout'])}"] - q[f"videos_at_{int(THRESHOLDS['legacy'])}"]
q["cumulative_incremental"] = q["incremental_videos"].cumsum()

ever55 = set(momentum_daily.loc[momentum_daily["momentum_score"]>=THRESHOLDS["pre_breakout"], "video_id"])
ever75 = set(momentum_daily.loc[momentum_daily["momentum_score"]>=THRESHOLDS["legacy"], "video_id"])
u55, u75 = len(ever55), len(ever75)

artists_55 = artists_75 = None
if "artist_name" in vids_safe.columns:
    m = vids_safe[["video_id","artist_name"]].drop_duplicates()
    artists_55 = m[m["video_id"].isin(ever55)]["artist_name"].nunique()
    artists_75 = m[m["video_id"].isin(ever75)]["artist_name"].nunique()

fig, (ax1, ax2) = plt.subplots(1,2, figsize=(16,7))

metrics, v55, v75 = ["Music Videos"], [u55], [u75]
if artists_55 is not None:
    metrics += ["Artists"]; v55 += [artists_55]; v75 += [artists_75]

x = np.arange(len(metrics)); w = 0.38
ax1.bar(x-w/2, v55, w, color=BLUE)
ax1.bar(x+w/2, v75, w, color=ORNG)
add_value_labels(ax1)
ax1.set_xticks(x); ax1.set_xticklabels(metrics)
ax1.set_ylabel("Count", color=TXT)
clean_title(ax1, f"Lowering to {THRESHOLDS['pre_breakout']} unlocks +{u55-u75:,} videos → more shots on goal")

ax2.plot(q["date"], q["cumulative_incremental"], linewidth=2.8, color=BLUE)
ax2.fill_between(q["date"], 0, q["cumulative_incremental"], color=BLUE, alpha=0.18)
ensure_year_on_dates(ax2)
ax2.set_ylabel("Cumulative Additional Video-Days", color=TXT)
clean_title(ax2, f"Threshold {THRESHOLDS['pre_breakout']} adds {int(q['incremental_videos'].sum()):,} video-days → align ops to activate")

plt.tight_layout(); plt.show()
print("✅ Chart 2.1 rendered")

## Chart 2.3 — Momentum Bar Race (weekly)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 11 — Chart 1.2: Like-rate shows content quality → interactive scatter by artist
# Hover to see video titles; color = artist; size = view count
# ═══════════════════════════════════════════════════════════════════════════
try:
    import plotly.express as px
    import plotly.graph_objects as go  # noqa: F401 (reserved for extension)
    from plotly.subplots import make_subplots  # noqa: F401 (reserved for extension)
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False
    print("⚠️  Plotly not installed. Run: pip install plotly")

if PLOTLY_AVAILABLE:
    df = vids_safe.copy()

    if "like_rate" not in df.columns:
        df["like_rate"] = (df["like_count"] / df["view_count"].replace(0, np.nan)).fillna(0.0).clip(0, 1)

    if "title" not in df.columns:
        df["title"] = df.get("video_title", df.get("video_id", "Unknown"))

    required_cols = ["pub_date", "like_rate", "artist_name", "view_count", "title"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise RuntimeError(f"Missing required columns for Chart 1.2: {missing}")

    top_artists = (df.groupby("artist_name").size()
                     .sort_values(ascending=False).head(6).index.tolist())
    df_plot = df[df["artist_name"].isin(top_artists)].copy()

    df_plot["hover_text"] = (
        "<b>" + df_plot["title"].astype(str).str[:70] + "</b><br>" +
        "Artist: " + df_plot["artist_name"].astype(str) + "<br>" +
        "Like Rate: " + (df_plot["like_rate"] * 100).round(1).astype(str) + "%<br>" +
        "Views: " + df_plot["view_count"].apply(lambda x: f"{x:,}") + "<br>" +
        "Date: " + df_plot["pub_date"].dt.strftime("%b %d, %Y")
    )

    fig = px.scatter(
        df_plot,
        x="pub_date",
        y="like_rate",
        color="artist_name",
        size="view_count",
        hover_data={"hover_text": True, "pub_date": False, "like_rate": False,
                    "artist_name": False, "view_count": False},
        labels={"like_rate": "Like Rate", "pub_date": "Release Date"},
        title=(
            "Like-rate shows content quality → higher likes = audience values the work"
            "<br><sub>Hover for video titles | Size = views | Color = artist</sub>"
        ),
        color_discrete_sequence=px.colors.qualitative.Bold,
        size_max=24,
    )

    fig.update_yaxes(tickformat=".1%", title="Like Rate (%)")
    fig.update_xaxes(title="Release Date")

    fig.update_traces(hovertemplate="%{customdata[0]}<extra></extra>")

    fig.update_layout(
        height=600,
        font=dict(size=12, color=TXT),
        plot_bgcolor="white",
        paper_bgcolor="white",
        legend=dict(
            title="Artist",
            orientation="v",
            yanchor="top",
            y=1,
            xanchor="left",
            x=1.02,
            bgcolor="rgba(255,255,255,0.85)",
            bordercolor="#cccccc",
            borderwidth=1
        ),
        hovermode="closest"
    )

    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor="#e8e8e8")
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor="#e8e8e8")

    fig.show()
    print("✅ Chart 1.2: Interactive like-rate scatter (hover for video titles)")

else:
    print("📊 Falling back to matplotlib version...")
    df = vids_safe.copy()
    if "like_rate" not in df.columns:
        df["like_rate"] = (df["like_count"] / df["view_count"].replace(0, np.nan)).fillna(0.0).clip(0,1)

    wk = (df.set_index("pub_date")
            .groupby("artist_name")["like_rate"]
            .resample("W").median()
            .reset_index())

    top_artists = (wk.groupby("artist_name")["like_rate"].count()
                     .sort_values(ascending=False).head(6).index.tolist())
    wk = wk[wk["artist_name"].isin(top_artists)]

    fig, ax = plt.subplots(figsize=(14,6))
    for artist, g in wk.groupby("artist_name"):
        g = g.sort_values("pub_date")
        y = (g["like_rate"]*100).values
        ax.plot(g["pub_date"], y, linewidth=2.2, label=artist, marker='o', markersize=4)

    ensure_year_on_dates(ax)
    ax.set_ylabel("Like Rate (%)", color=TXT, fontsize=11)
    ax.legend(title="Artist", loc="upper left", frameon=True, fancybox=False,
              edgecolor="#cccccc", fontsize=10)
    clean_title(ax, "Like-rate shows content quality → higher likes = audience values the work (weekly median)")
    ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
    print("✅ Chart 1.2: Like-rate by artist (matplotlib fallback)")

## KPI‑22 — Breakout Duration & Pre‑Warning (Top‑10 by artist; cap‑aware)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 16 — KPI-22: Breakout duration (days) + Pre-warning (hours)
# • Breakout episode = contiguous days with score ≥ BRK (default 60)
# • Pre-warning window = contiguous days immediately before start with PRE≤score<BRK
# • Visual: top-10 artists (mean duration | mean pre-warning), direct-labeled barh
# • Cap handling: pre-warning capped at CAP_HOURS, hatched + annotated
# ═══════════════════════════════════════════════════════════════════════════
PRE, BRK, CAP_HOURS = 55, 60, 720  # 30 days cap

if "momentum_daily" not in globals():
    raise RuntimeError("momentum_daily missing.")

md = momentum_daily.copy()
md["date"] = pd.to_datetime(md["date"]).dt.floor("D")
md = md.sort_values(["video_id","date"])

# Map video -> artist for labeling
va = vids_safe[["video_id","artist_name"]].drop_duplicates() if "artist_name" in vids_safe.columns else None
def _artist_of(vid):
    if va is None:
        return "unknown"
    row = va.loc[va["video_id"] == vid]
    return row["artist_name"].iloc[0] if not row.empty else "unknown"

# Build fast lookup of scores per video/date
episodes = []
for vid, g in md.groupby("video_id", sort=False):
    g2 = g[["date","momentum_score"]].sort_values("date").set_index("date")
    dates = g2.index.tolist()
    in_ep = False
    start = None
    for i, d in enumerate(dates):
        s = float(g2.loc[d, "momentum_score"])
        if s >= BRK and not in_ep:
            in_ep = True
            start = d
        elif (s < BRK or i == len(dates)-1) and in_ep:
            end = dates[i] if (s >= BRK and i == len(dates)-1) else dates[i-1]
            pre_days = 0
            step = 1
            while True:
                prev_day = start - pd.Timedelta(days=step)
                if prev_day not in g2.index:
                    break
                prev_s = float(g2.loc[prev_day, "momentum_score"])
                if PRE <= prev_s < BRK:
                    pre_days += 1
                    step += 1
                else:
                    break
            episodes.append({
                "video_id": vid,
                "artist_name": _artist_of(vid),
                "start": start,
                "end": end,
                "duration_days": int((end - start).days + 1),
                "pre_warning_hours": int(min(pre_days * 24, CAP_HOURS)),
                "capped": pre_days * 24 >= CAP_HOURS,
            })
            in_ep = False

episodes_df = pd.DataFrame(episodes)
if episodes_df.empty:
    print("⚠️  KPI-22: no breakouts detected (score ≥ 60).")
else:
    agg = (
        episodes_df.groupby("artist_name")
        .agg(
            mean_duration_days=("duration_days", "mean"),
            mean_pre_warning_hours=("pre_warning_hours", "mean"),
            any_capped=("capped", "max"),
        )
        .reset_index()
    )

    top_dur = agg.nlargest(10, "mean_duration_days").sort_values("mean_duration_days")
    top_warn = agg.nlargest(10, "mean_pre_warning_hours").sort_values("mean_pre_warning_hours")

    overall_dur = agg["mean_duration_days"].mean()
    overall_warn = agg["mean_pre_warning_hours"].mean()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7), sharex=False)

    # Left: duration
    y = np.arange(len(top_dur))
    bars_dur = ax1.barh(y, top_dur["mean_duration_days"], color=ORNG, alpha=0.9, height=0.6)
    ax1.set_yticks(y)
    ax1.set_yticklabels(top_dur["artist_name"], fontsize=10)
    ax1.set_xlabel("Avg Breakout Duration (days)", fontsize=11)
    ax1.set_title(
        f"KPI-22 (1/2): Breakouts last {overall_dur:.1f} days on average\n→ plan sustain tactics",
        fontweight="bold",
        fontsize=12,
        color=TXT,
        pad=12,
    )
    add_value_labels(ax1, fmt="{:.1f}", orientation="horizontal")
    ax1.grid(True, axis="x", alpha=0.3, color=GRID)

    # Right: pre-warning (cap-aware)
    y2 = np.arange(len(top_warn))
    bars_warn = ax2.barh(y2, top_warn["mean_pre_warning_hours"], color=PURP, alpha=0.9, height=0.6)
    cap_lookup = agg.set_index("artist_name")["any_capped"].to_dict()
    for idx, artist in enumerate(top_warn["artist_name"]):
        if cap_lookup.get(artist, False):
            bars_warn[idx].set_hatch("///")
    ax2.set_yticks(y2)
    ax2.set_yticklabels(top_warn["artist_name"], fontsize=10)
    ax2.set_xlabel("Avg Pre-Breakout Warning (hours)", fontsize=11)
    cap_note = "\n(hatched bars capped at ≥720h)" if any(cap_lookup.values()) else ""
    ax2.set_title(
        f"KPI-22 (2/2): Pre-breakout window ≈ {overall_warn:.0f}h on avg\n→ act inside window{cap_note}",
        fontweight="bold",
        fontsize=12,
        color=TXT,
        pad=12,
    )
    add_value_labels(ax2, fmt="{:.0f}", orientation="horizontal")
    ax2.grid(True, axis="x", alpha=0.3, color=GRID)

    plt.tight_layout()
    plt.show()
    print("✅ KPI-22 rendered (top-10 bars, contiguous logic, cap-aware)")

## FYI — Sentiment (diverging 100% bars)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# VISUALIZATION HELPERS (Professional Standards)
# ═══════════════════════════════════════════════════════════════════════════

# --- Accessible Color Palette (4.5:1 contrast, no red/green combos) ---
PALETTE = [
    "#1b9e77",  # teal (primary signal)
    "#d95f02",  # orange (secondary signal)
    "#7570b3",  # purple (tertiary)
    "#e7298a",  # magenta (accent)
    "#66a61e",  # olive (context)
    "#e6ab02",  # gold (highlight)
]
GREY_1 = "#cccccc"  # light grey (context, non-data)
GREY_2 = "#888888"  # medium grey (neutral sentiment)
GREY_3 = "#444444"  # dark grey (text)
BLUE_BREAKOUT = "#3498db"  # BLUE for pre-breakout state (≥55)

def get_color(i: int) -> str:
    return PALETTE[i % len(PALETTE)]

def slide(figsize=(11,6), watermark: Optional[str]=None):
    """Create a professional slide-style figure."""
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_anchor("NW")
    if watermark:
        fig.text(0.5, 0.5, watermark, color=GREY_1, fontsize=60, ha="center",
                 va="center", alpha=0.35, rotation=30)
    return fig, ax

def action_title(ax: mpl.axes.Axes, finding: str, implication: str, action: str) -> None:
    """Set action-oriented title: Finding → Implication → Action."""
    ax.set_title(f"{finding} → {implication} → {action}", fontsize=13, fontweight="bold", pad=12)

def direct_line_labels(ax: mpl.axes.Axes, fontsize: int = 10):
    """Add direct labels to line chart (remove legend)."""
    lines = [ln for ln in ax.get_lines() if not ln.get_label().startswith("_")]
    for ln in lines:
        x, y = ln.get_xdata(), ln.get_ydata()
        if len(x)==0: continue
        ax.annotate(ln.get_label(), xy=(x[-1], y[-1]), xytext=(5,0), textcoords="offset points",
                    va="center", fontsize=fontsize, color=ln.get_color(), fontweight="bold")
    if ax.get_legend(): ax.get_legend().remove()

def label_bars(ax: mpl.axes.Axes, fmt="{:.0f}", fontsize=10):
    """Add value labels to bar chart."""
    for p in ax.patches:
        h = p.get_height()
        if h == 0: continue
        ax.text(p.get_x()+p.get_width()/2, p.get_y()+h, fmt.format(h),
                ha="center", va="bottom", fontsize=fontsize, color=GREY_3)

def iso8601_to_seconds(iso: str) -> int:
    """Parse ISO 8601 duration (PT1H2M3S) to seconds."""
    if not isinstance(iso, str): return 0
    h = m = s = 0
    mobj = re.match(r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?", iso)
    if mobj:
        h = int(mobj.group(1) or 0); m = int(mobj.group(2) or 0); s = int(mobj.group(3) or 0)
    return h*3600 + m*60 + s

print('✅ Visualization helpers loaded')

## FYI — Publish Hour vs Avg Views/Day

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# DATA PREPARATION + SAFE COPIES
# ═══════════════════════════════════════════════════════════════════════════

# Create working copies
vids = videos_df.copy()
comments = comments_df.copy()

# Derived video features
vids["age_days"] = (END_DATE - vids["published_at"]).dt.days.clip(lower=1)
vids["views_per_day"] = (vids["view_count"] / vids["age_days"]).replace([np.inf, np.nan], 0.0)
vids["like_rate"] = (vids["like_count"] / vids["view_count"].replace(0, np.nan)).fillna(0.0).clip(0,1)
vids["comment_rate"] = (vids["comment_count"] / vids["view_count"].replace(0, np.nan)).fillna(0.0).clip(0,1)
vids["publish_week"] = vids["published_at"].dt.to_period("W").dt.to_timestamp()
vids["publish_month"] = vids["published_at"].dt.to_period("M").dt.to_timestamp()
vids["publish_hour"] = vids["published_at"].dt.hour
vids["publish_dow"] = vids["published_at"].dt.day_name()

# Comment features
if 'text' in comments.columns:
    comments["comment_length"] = comments["text"].str.len().fillna(0)

# ── Safe copies for downstream visuals ──────────────────────────────────────
vids_safe = vids.copy()
vids_safe["published_at"] = pd.to_datetime(vids_safe["published_at"], errors="coerce")
vids_safe = vids_safe.dropna(subset=["published_at"])
vids_safe["pub_date"] = vids_safe["published_at"].dt.floor("D")

comments_safe = None
if "comments_df" in globals():
    comments_safe = comments_df.copy()
elif "comments" in globals():
    comments_safe = comments.copy()

if comments_safe is not None:
    comments_safe["published_at"] = pd.to_datetime(comments_safe["published_at"], errors="coerce")
    comments_safe = comments_safe.dropna(subset=["published_at"])
    comments_safe["pub_date"] = comments_safe["published_at"].dt.floor("D")

print(f"✅ Data prepared: {len(vids):,} videos with derived features")
print(f"   Age range: {vids['age_days'].min():.0f} - {vids['age_days'].max():.0f} days")
print(f"   Views/day range: {vids['views_per_day'].min():.1f} - {vids['views_per_day'].max():.1f}")
print("✅ Metrics prepared (comments optional).")

## FYI — Comment Length Distribution

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# DATA LOADING & VALIDATION
# ═══════════════════════════════════════════════════════════════════════════

# Discover and load data from YouTube API v3
chart_data = discover_data()

print('📈 Data Summary:')
for data_type, df in chart_data.items():
    print(f'   {data_type}: {len(df):,} rows, {len(df.columns)} columns')
    if 'artist_name' in df.columns:
        unique_artists = df['artist_name'].nunique()
        print(f'      → {unique_artists} unique artists')

# Extract core DataFrames
videos_df = chart_data.get('videos', pd.DataFrame())
comments_df = chart_data.get('comments', pd.DataFrame())
metrics_df = chart_data.get('metrics_timeseries', pd.DataFrame())

# Fix column name mismatch BEFORE validation
if 'comment_text' in comments_df.columns and 'text' not in comments_df.columns:
    comments_df['text'] = comments_df['comment_text']

# --- Data Contract Validation ---
REQUIRED_VIDEO_COLS = ['video_id', 'title', 'artist_name', 'published_at', 'view_count', 'like_count', 'comment_count']
REQUIRED_COMMENT_COLS = ['video_id', 'published_at', 'text']

missing_video_cols = set(REQUIRED_VIDEO_COLS) - set(videos_df.columns)
missing_comment_cols = set(REQUIRED_COMMENT_COLS) - set(comments_df.columns)

if missing_video_cols:
    raise ValueError(f"❌ Missing required video columns: {missing_video_cols}")
if missing_comment_cols:
    raise ValueError(f"❌ Missing required comment columns: {missing_comment_cols}")

# Ensure datetime types
videos_df['published_at'] = pd.to_datetime(videos_df['published_at'])
comments_df['published_at'] = pd.to_datetime(comments_df['published_at'])

# Determine artist roster
if ARTISTS_OVERRIDE:
    artists = ARTISTS_OVERRIDE
    print(f"\n🎵 Using override roster: {len(artists)} artists")
else:
    artists = sorted(videos_df['artist_name'].unique())
    print(f"\n🎵 Discovered roster: {len(artists)} artists")

for i, artist in enumerate(artists, 1):
    print(f"   {i}. {artist}")

ARTIST_COUNT = len(artists)

print(f"\n✅ Data loaded: {len(videos_df):,} videos, {len(comments_df):,} comments from {ARTIST_COUNT} artists")

## Data Provenance
This dashboard uses YouTube Data API v3 metrics gathered by our ETL. If synthetic or subset data is used, labels on charts will indicate it.

## Quality Helpers (appendix)
- Direct labels; legends avoided
- Use grey for context; ≤10% highlight color
- Modified‑Z outliers (|z|>3.5)
- Contrast ≥4.5:1 for text
- Year always visible on date axes